<a href="https://colab.research.google.com/github/ntatfff/todo/blob/202411241258/ColabRadiomicsFeatureExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
kernel = 1
className = 'firstorder'

In [2]:
# Parameters
kernel = 3
className = "ngtdm"


In [3]:
import radiomics
import numpy as np
import SimpleITK as sitk
import radiomics.featureextractor
import os
import six
from os import path
import pandas as pd

def featureExtractor(fileId):
  imagePath = './dataset/BraTS2021_Training_Data/%s/%s_flair.nii.gz' % (fileId, fileId)
  image = sitk.ReadImage(imagePath)
  maskPath ='./dataset/BraTS2021_Training_Data/%s/%s_kernel%s_tumor.nii.gz' % (fileId, fileId, kernel)
  mask = sitk.ReadImage(maskPath)

  settings = {}
  settings['kernelRadius'] = kernel
  settings['maskedKernel'] = False
  settings['voxelBatch'] = 1000
  extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(**settings)
  extractor.disableAllFeatures()
  extractor.enableFeatureClassByName(className)

  featureMap = extractor.execute(image, mask, voxelBased=True)

  for featureName, featureValue in six.iteritems(featureMap):
    if isinstance(featureValue, sitk.Image):
      fileFolder = './dataset/%s/kernel%s/tumor/%s' % (className, kernel, fileId)
      if path.exists(fileFolder) == False:
        os.makedirs(fileFolder, exist_ok=True)
      sitk.WriteImage(featureValue, '%s/%s.nrrd' % (fileFolder, featureName))
      print('Computed %s, stored as "%s/%s.nrrd"' % (featureName, fileFolder, featureName))
    # else:
    #   print('%s: %s' % (featureName, featureValue))

monitorFilePath = './dataset/%s/kernel%s/tumor.monitor.csv' % (className, kernel)
monitor = pd.read_csv(monitorFilePath, index_col='no')
for i, row in monitor.iterrows():
  if row['done'] != 0:
    continue
  fileId = row['file']
  print('Starting %s' % (fileId))
  featureExtractor(fileId)
  monitor.at[i, 'done'] = 1
  monitor.to_csv(monitorFilePath)

Starting BraTS2021_00002


Computed original_ngtdm_Busyness, stored as "./dataset/ngtdm/kernel3/tumor/BraTS2021_00002/original_ngtdm_Busyness.nrrd"
Computed original_ngtdm_Coarseness, stored as "./dataset/ngtdm/kernel3/tumor/BraTS2021_00002/original_ngtdm_Coarseness.nrrd"
Computed original_ngtdm_Complexity, stored as "./dataset/ngtdm/kernel3/tumor/BraTS2021_00002/original_ngtdm_Complexity.nrrd"
Computed original_ngtdm_Contrast, stored as "./dataset/ngtdm/kernel3/tumor/BraTS2021_00002/original_ngtdm_Contrast.nrrd"
Computed original_ngtdm_Strength, stored as "./dataset/ngtdm/kernel3/tumor/BraTS2021_00002/original_ngtdm_Strength.nrrd"
